In [2]:
# basic idea is to split one document (for example) into larger chunks and then split these chunks into smaller chunks. You then embed these smaller chunks
# into a vector store. So you get accurate retrieval, plus enough context. 

# for tmo bonds use case:
# 1. larger chunks of meaningful components like topics, smaller chunks of meaningful components
# 2. larger chunks of fixed length, smaller chunks of fixed length
# 3. larger chunks of meaningful components, smaller chunks of fixed length

In [13]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-4", temperature=0)
from langchain.retrievers import ParentDocumentRetriever

In [14]:
# note: use openai == 0.28.1

In [15]:
import os
import openai

In [16]:
llm.predict("hello")

'Hello! How can I assist you today?'

In [37]:
loaders = [
    TextLoader("./paul_graham_essay.txt")
    #TextLoader("../../state_of_the_union.txt"),
]
docs = []
for l in loaders:
    docs.extend(l.load())

In [38]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# This text splitter is used to create the child documents
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# The storage layer for the parent documents
store = InMemoryStore()

In [39]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [40]:
retriever.add_documents(docs)

In [41]:
search = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever
)

In [42]:
answer = search("What was paul's favorite burger?")
print(answer)

{'query': "What was paul's favorite burger?", 'result': "Paul's favorite burger was Chicken McSpicy."}


In [43]:
print(answer['result'])

Paul's favorite burger was Chicken McSpicy.
